import libraries

In [1]:
import re
import numpy as np
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util

MODEL_NAME = "all-MiniLM-L6-v2"
HF_DATASET = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"
PER_INTENT = 150            # rows kept per intent (keeps the dataset medium-sized)
FAQ_THRESHOLD = 0.55        # min similarity to trust an FAQ answer
TICKET_THRESHOLD = 0.50     # min similarity to trust a ticket answer
FALLBACK = ("Sorry, I'm not sure about that. Please contact your mentor or "
            "the support team and they will help you.")

to load the datset


In [2]:
from google.colab import files

uploaded = files.upload()
FAQ_FILE = list(uploaded.keys())[0]   # works even if the file is renamed
print("Uploaded:", FAQ_FILE)

Saving internship_faq.csv to internship_faq.csv
Uploaded: internship_faq.csv


In [3]:
def clean_text(text):
    """Turn placeholders like {{Order Number}} into [Order Number]."""
    return re.sub(r"\{\{(.*?)\}\}", r"[\1]", str(text)).strip()


def load_tickets():
    """Historical support tickets from the public Bitext dataset."""
    df = load_dataset(HF_DATASET, split="train").to_pandas()
    df = df[["instruction", "response", "intent"]]
    df.columns = ["question", "answer", "intent"]
    df = df.drop_duplicates(subset="question")
    df = df.sample(frac=1, random_state=42)             # shuffle
    df = df.groupby("intent").head(PER_INTENT)          # max rows per intent
    df["answer"] = df["answer"].apply(clean_text)
    return df.reset_index(drop=True)


def load_faq():
    """FAQ documents from the uploaded CSV."""
    df = pd.read_csv(FAQ_FILE)
    df["intent"] = "internship_faq"
    return df


model = SentenceTransformer(MODEL_NAME)
tickets = load_tickets()
faq = load_faq()
print(f"Tickets: {len(tickets)} rows | FAQ: {len(faq)} rows")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

Tickets: 4050 rows | FAQ: 55 rows


In [4]:
def encode(texts):
    return model.encode(texts, convert_to_numpy=True,
                        normalize_embeddings=True, show_progress_bar=True)

faq_emb = encode(faq["question"].tolist())
tick_emb = encode(tickets["question"].tolist())


def best_match(query, df, emb):
    q = model.encode(query, convert_to_numpy=True, normalize_embeddings=True)
    scores = util.cos_sim(q, emb)[0].numpy()
    i = int(scores.argmax())
    return df.iloc[i], float(scores[i])


def get_reply(query):
    """FAQ first, then historical tickets, then fallback."""
    row, score = best_match(query, faq, faq_emb)
    if score >= FAQ_THRESHOLD:
        return row["answer"], "FAQ", score
    row, score = best_match(query, tickets, tick_emb)
    if score >= TICKET_THRESHOLD:
        return row["answer"], f"ticket:{row['intent']}", score
    return FALLBACK, "fallback", score


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/127 [00:00<?, ?it/s]

In [ ]:
print("Internship Support chatbot ready. Type 'quit' to exit.\n")
while True:
    query = input("You: ").strip()
    if query.lower() in {"quit", "exit", "q"}:
        print("chatBot: Good luck with your internship!")
        break
    if not query:
        continue
    answer, source, score = get_reply(query)
    print(f"chatBot: {answer}\n     [{source} | similarity {score:.2f}]\n")

Internship Support chatbot ready. Type 'quit' to exit.

